In [1]:
!pip install google-adk --quiet

In [2]:
import os
import logging

import vertexai
from google.adk.agents import LlmAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.loop_agent import LoopAgent
from google.adk.tools.google_search_tool import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-00-11b9d8ae6979"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

GEMINI_MODEL = "gemini-2.5-flash"

In [17]:
greeter_agent = LlmAgent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description="Greets the user and clarifies their question.",
    instruction="""
    You are a friendly greeter. Your job is to take the user's question
    and rephrase it as a clear, specific research query.

    Output ONLY the rephrased research query. No greeting, no preamble,
    no commentary. Just the query itself.

    Examples:
    - User: "Who won the Super Bowl?" → "Who won the most recent Super Bowl and what was the final score?"
    - User: "What's new in fusion?" → "What are the most significant recent breakthroughs in nuclear fusion energy?"
    """,
    output_key="user_question",
)

In [18]:
search_agent = LlmAgent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Searches Google for current information to answer the question.",
    instruction="""
    You are a research agent with access to Google Search.

    Research the following question:
    {user_question}

    Use the google_search tool to find accurate, up-to-date information.
    Provide a thorough initial answer based on the search results.
    Always base your answer on search results, not your own training data.
    Include specific facts, dates, and details from the search results.
    """,
    tools=[google_search],
    output_key="current_answer",
)

In [19]:
critique_agent = LlmAgent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Reviews an answer and provides constructive critique.",
    instruction="""
    You are an expert editor. Review the following answer to the question:
    {user_question}

    **Answer to review:**
    {current_answer}

    **IMPORTANT: The answer was produced by a live Google Search and contains
    real, current information. Do NOT question the factual accuracy of search
    results based on your own training data. Your training data may be outdated.
    Treat all search-sourced facts as correct.**

    **Evaluate ONLY against these criteria:**
    1. Completeness: Does it fully address the question?
    2. Clarity: Is the answer clear and well-organized?
    3. Conciseness: Is it an appropriate length without unnecessary filler?
    4. Specificity: Does it include concrete details (dates, numbers, names)?
    5. Readability: Is it easy for a general audience to understand?

    **Output:** A concise bulleted list of specific improvements needed.
    If the answer is excellent, state: "No major issues found."
    """,
    output_key="critique_notes",
)

In [20]:
refine_agent = LlmAgent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Refines an answer based on critique feedback.",
    instruction="""
    You are an expert editor. Improve the following answer based on the
    critique provided.

    **Original question:** {user_question}

    **Current answer:**
    {current_answer}

    **Critique feedback:**
    {critique_notes}

    **Task:**
    Rewrite the answer to address every point raised in the critique.
    If the critique says "No major issues found," return the answer as-is.
    Make the answer clear, accurate, complete, and well-organized.

    Output ONLY the improved answer text. No preamble or commentary.
    """,
    output_key="current_answer",  # Overwrites for next loop iteration
)

In [21]:
# Critique-Refine loop: iteratively improves the answer
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critique_agent, refine_agent],
    max_iterations=2,
    description="Iteratively critiques and refines the answer.",
)

# Full pipeline: greet → search → refine loop
answer_workflow = SequentialAgent(
    name="answer_workflow",
    sub_agents=[greeter_agent, search_agent, refinement_loop],
    description="End-to-end workflow that researches a question, then iteratively refines the answer.",
)

root_agent = answer_workflow

In [22]:
async def run_workflow(user_input: str) -> str:
    """Send a question through the full answer workflow and return the final response.
    Prints events showing each agent in the pipeline as it executes.

    Args:
        user_input (str): The user's question.

    Returns:
        str: The final refined answer from state.
    """
    runner = InMemoryRunner(agent=root_agent, app_name="workflow_app")
    session = await runner.session_service.create_session(
        app_name="workflow_app", user_id="user_01"
    )
    message = types.Content(role="user", parts=[types.Part(text=user_input)])

    current_agent = None
    print(f"\n{'='*60}")
    print(f"Question: {user_input}")
    print(f"{'='*60}")

    for event in runner.run(
        user_id="user_01",
        session_id=session.id,
        new_message=message,
    ):
        # Track agent transitions
        if event.author and event.author != current_agent:
            current_agent = event.author
            print(f"\n  >> Agent: {current_agent}")

        # Show function calls (e.g., google_search)
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    print(f"     [TOOL CALL] {part.function_call.name}")
                if hasattr(part, "function_response") and part.function_response:
                    print(f"     [TOOL RESPONSE] ← {part.function_response.name}")

    # Read the final answer from shared state — not from event stream
    final_session = await runner.session_service.get_session(
        app_name="workflow_app", user_id="user_01", session_id=session.id
    )
    state = final_session.state

    print(f"\n{'─'*60}")
    print("STATE after workflow:")
    print(f"  user_question:  {str(state.get('user_question', ''))[:100]}...")
    print(f"  current_answer: {str(state.get('current_answer', ''))[:150]}...")
    print(f"  critique_notes: {str(state.get('critique_notes', ''))[:150]}...")
    print(f"{'─'*60}")

    return state.get("current_answer", "No answer produced.")

In [23]:
from IPython.display import Markdown, display

test_queries = [
    "Who won the most recent Super Bowl and what was the final score?",
    "What are the biggest recent breakthroughs in nuclear fusion energy?",
    "What major tech layoffs have happened in 2025?",
]

print("=" * 60)
print("CHALLENGE 4 — Agent Workflow Test")
print("Sequential Agent → Loop Agent (Critique + Refine)")
print("=" * 60)

for query in test_queries:
    response = await run_workflow(query)
    display(Markdown(f"**Final Answer:**\n\n{response}"))
    print("\n" + "=" * 60)

CHALLENGE 4 — Agent Workflow Test
Sequential Agent → Loop Agent (Critique + Refine)

Question: Who won the most recent Super Bowl and what was the final score?

  >> Agent: greeter_agent

  >> Agent: search_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

────────────────────────────────────────────────────────────
STATE after workflow:
  user_question:  Who won the most recent Super Bowl and what was the final score?...
  current_answer: The most recent Super Bowl, Super Bowl LX, was won by the Seattle Seahawks. They defeated the New England Patriots with a final score of 29-13 on Febr...
  critique_notes: No major issues found....
────────────────────────────────────────────────────────────


**Final Answer:**

The most recent Super Bowl, Super Bowl LX, was won by the Seattle Seahawks. They defeated the New England Patriots with a final score of 29-13 on February 8, 2026.



Question: What are the biggest recent breakthroughs in nuclear fusion energy?

  >> Agent: greeter_agent

  >> Agent: search_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

────────────────────────────────────────────────────────────
STATE after workflow:
  user_question:  What are the most significant recent breakthroughs in nuclear fusion energy?...
  current_answer: Recent breakthroughs in nuclear fusion energy have significantly advanced the field, bringing the prospect of a clean and virtually limitless power so...
  critique_notes: *   **Clarify "integrated energy turnover" for Wendelstein 7-X:** While technically correct, "integrated energy turnover in the plasma" might not be i...
────────────────────────────────────────────────────────────


**Final Answer:**

Recent breakthroughs in nuclear fusion energy have significantly advanced the field, bringing the prospect of a clean and virtually limitless power source closer to reality. These advancements span both inertial and magnetic confinement fusion, with notable achievements in energy gain, plasma stability, and increased private and public investment.

One of the most significant breakthroughs has been the repeated achievement of "net energy gain" in inertial confinement fusion. The U.S. Lawrence Livermore National Laboratory's (LLNL) National Ignition Facility (NIF) first accomplished this in December 2022, where the energy produced from the fusion reaction surpassed the laser energy delivered to the target. This was replicated in August 2023, and subsequent reports indicated that by April 2025, the NIF team had delivered 8.6 megajoules (MJ) of fusion energy from 2.08 MJ of laser input, more than quadrupling the input energy for ignition. This demonstrates a substantial improvement in the efficiency of laser-powered fusion.

In magnetic confinement fusion, progress has been made across various experimental reactors:
*   **Stellarators:** Germany's Wendelstein 7-X stellarator achieved an integrated energy turnover in the plasma of 1.3 gigajoules in 2023 (a measure of the total thermal energy handled by the plasma over time, indicating significant plasma performance and confinement), a record for this technology, and maintained plasma for eight minutes. By May 2025, its integrated energy turnover in the plasma was further raised to 1.8 gigajoules, demonstrating improved plasma performance and duration.
*   **Tokamaks:** China's "artificial sun" reactor has also broken long-standing thresholds in fusion technology. In May 2021, its Experimental Advanced Superconducting Tokamak (EAST) set a world record by achieving a plasma temperature of 120 million degrees Celsius for 101 seconds and 160 million degrees Celsius for 20 seconds. Other tokamaks, including the WEST reactor in France and KSTAR in South Korea, have also broken records for plasma duration and stability.
*   **High-Temperature Superconducting (HTS) Magnets:** The development and application of HTS magnets are revolutionizing fusion reactor design, allowing for more compact and efficient machines. Projects like SPARC and WHAM are integrating HTS coils to enhance performance and reduce size, cost, and development time.

The private sector has become a major driver of innovation and investment in fusion energy. Companies have raised approximately $5 billion in private funding, with total private investment reaching nearly $9 billion between 2021 and 2025. Notably:
*   **Helion Energy:** This Washington-based startup achieved plasma temperatures of 150 million degrees Celsius (10 times hotter than the Sun) and successfully fused hydrogen isotopes deuterium and tritium. Microsoft announced a deal in May 2023 to purchase fusion-generated electricity from Helion by 2028.
*   **Commonwealth Fusion Systems (CFS):** Backed by investors including Bill Gates, CFS is leading in private funding, having raised over $2 billion. They plan to put fusion power on the grid in the early 2030s, utilizing high-temperature superconducting tape to create strong magnetic fields for plasma confinement.
*   **OpenStar:** This New Zealand-based company demonstrated the levitation of a half-ton superconducting magnet while confining plasma, a crucial step for its unique levitated dipole method.

International collaboration continues to be vital, with projects like ITER (International Thermonuclear Experimental Reactor) in France aiming to be the world's largest tokamak. While construction has faced delays, research operations are now slated to begin in 2034, with full-scale fusion reactions targeted for 2039. China has also significantly ramped up its fusion research, with an estimated annual budget of around $1.5 billion, nearly double that allocated by the U.S. government in 2024.

Further technological advancements include the application of Artificial Intelligence (AI) and machine learning to predict and prevent plasma instabilities in tokamak reactors, which can disrupt the fusion reaction. This increases confidence in operating these complex machines without issues.

In addition to scientific and technological progress, regulatory changes in the U.S. in 2023 have streamlined the process for deploying fusion energy, treating fusion reactors more like particle accelerators than traditional nuclear fission plants, which are considered less risky. This regulatory clarity is expected to accelerate commercialization efforts.



Question: What major tech layoffs have happened in 2025?

  >> Agent: greeter_agent

  >> Agent: search_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

  >> Agent: critique_agent

  >> Agent: refine_agent

────────────────────────────────────────────────────────────
STATE after workflow:
  user_question:  What major tech companies have announced or experienced significant layoffs in 2025?...
  current_answer: In 2025, the tech industry experienced significant layoffs across numerous major companies, with over 127,000 workers at U.S.-based tech companies los...
  critique_notes: No major issues found....
────────────────────────────────────────────────────────────


**Final Answer:**

In 2025, the tech industry experienced significant layoffs across numerous major companies, with over 127,000 workers at U.S.-based tech companies losing their jobs and global figures exceeding 150,000. This trend was largely driven by strategic shifts towards AI and automation, cost-cutting initiatives, and organizational restructuring aimed at enhancing efficiency and profitability.

Major tech companies that announced or experienced substantial layoffs in 2025 include:

*   **Intel:** The chipmaking giant announced plans to cut approximately 24,500 employees, aiming to reduce its core workforce to about 75,000 by the end of the year. This represents about one-quarter of its workforce, with layoffs occurring in multiple rounds throughout the year, including 15 percent of its factory workers in July and around 100 in Chandler, Arizona, after Labor Day.
*   **Microsoft:** Microsoft implemented multiple rounds of layoffs, totaling around 15,000 employees in 2025. This included about 6,000 workers (3% of its workforce) in May and an additional 9,000 employees in July. These cuts impacted various business units, including sales, Xbox, middle management, and support functions, as the company redirected funds and talent toward AI infrastructure.
*   **Amazon:** Amazon announced significant job cuts, with over 14,000 corporate jobs eliminated, and some reports suggesting as many as 30,000 corporate roles could be impacted. Layoffs affected several groups within Amazon Web Services (AWS) and other departments, driven by the company's move to become leaner with fewer management layers and embrace AI.
*   **Block (formerly Square):** The fintech company announced a major reduction of over 4,000 employees, nearly half its global workforce, as it shifted to AI tools to enable smaller teams to operate more efficiently. CEO Jack Dorsey cited intelligence tools as a key factor in transforming how the company builds and runs operations.
*   **Salesforce:** The CRM giant cut approximately 4,000 customer support jobs and an additional 1,000 employees, with CEO Mark Benioff acknowledging AI as a significant reason. Salesforce simultaneously embarked on an AI sales position hiring spree.
*   **HP (Hewlett-Packard):** HP announced plans to lay off between 1,000 and 2,000 employees by October 2025 as part of a cost-cutting strategy aimed at saving $300 million. These reductions were part of the "Future Now" initiative to streamline operations and invest in AI-driven technologies.
*   **Verizon:** The carrier giant prepared to cut 15,000 jobs, or about 15 percent of its workforce, as part of restructuring efforts.
*   **IBM:** In March, IBM laid off employees in its communications and marketing division, continuing a trend of reducing non-customer-facing back-office roles that AI and automation can perform.
*   **Google:** Google steadily trimmed staff throughout 2025, laying off hundreds of employees across various divisions, including its Global Business Organization, Pixel, Android, Chrome, and cloud unit's design-related roles. These cuts were attributed to restructuring efforts, shifting focus, and automating roles, with Google ramping up investments in AI infrastructure.
*   **STMicroelectronics:** The company revealed plans for 5,000 roles to disappear over three years, including 2,800 announced layoffs, as it realigned operations with shifting semiconductor market trends.
*   **TomTom:** Announced 300 job cuts, about 10% of its staff, as it pivoted from standalone GPS devices to AI-powered mapping for vehicles.
*   **Bumble:** Laid off 240 employees, representing 30% of its workforce.
*   **Jamf:** The Apple device management software provider impacted over 6% of its staff in layoffs aimed at reducing operating costs and allowing for strategic reinvestment.
*   **HPE (Hewlett Packard Enterprise):** Announced plans to slash roughly 5% of its workforce, or 2,500 jobs, over 18 months as a cost-cutting measure to save $350 million by its 2027 fiscal year.
*   **TCS (Tata Consultancy Services):** One of the world's largest IT services providers, TCS announced plans to lay off 12,000 employees.

Overall, the first five months of 2025 alone saw over 62,000 employees laid off across 284 companies. By July 31, an estimated 80,250 tech employees had been laid off in 2025. This continued the trend from 2024, where 152,922 tech employees were laid off. Many of these layoffs, particularly at larger companies, were explicitly linked to the increasing adoption of AI and automation, economic uncertainties, and efforts to streamline operations and reallocate resources towards strategic areas.